# 1. The baseline TB model

> *"For one of the STRIDE projects, we'd potentially be looking at an intervention that gets
> triggered once prevalence reaches a given threshold. Is it possible to do this with summer?"*

**Yes.** This repository is a small, deliberately simple TB model that shows how, using the
latest summer4 release (`v0.2.0a5`). There are three notebooks:

1. **This one** builds the TB model step by step and runs the baseline. It is written for
   people who know summer2, so each step says what it replaces.
2. [`02-triggered-interventions`](02-triggered-interventions.ipynb) explains how the threshold
   trigger works and compares four intervention scenarios against the baseline.
3. [`03-thresholds-and-signals`](03-thresholds-and-signals.ipynb) sweeps the threshold, takes
   gradients through the trigger, and triggers on notifications instead of prevalence.

The model is illustrative. Its parameters are TB orders of magnitude, not a calibration to any
setting.

## summer2 to summer4, in one table

| summer2 | summer4 |
|---|---|
| `CompartmentalModel(times, compartments, infectious_compartments)` | `Property` + `PropertyMap` define the compartments; `FlowModel(pmap)` holds the flows |
| `model.stratify_with(Stratification(...))` | `pmap.stratify(prop, where=selector)`. Stratifying only part of the model is allowed ("ragged") |
| `add_infection_frequency_flow` | `TransitionFlow(name, source, dest, ForceOfInfection(...))` |
| `add_transition_flow` / `add_death_flow` / `add_universal_death_flows` | `TransitionFlow` / `ExitFlow` with any selector as source |
| `add_replacement_birth_flow` | `EntryFlow("birth", S, deaths.sum())`: births are the sum of the death flows |
| `Parameter("x")` | `Param("x")`; values come from a plain dict at run time |
| `get_sigmoidal_interpolation_function`, `get_linear_interpolation_function` | `summer4.timevarying.sigmoidal` / `linear` on `Time()` |
| Function flows / computed values that read the model state | `Reduce(...)` reads compartment sizes; `defer(fn)(...)` wraps any JAX function |
| `request_output_for_compartments` / `request_output_for_flow` | `SavePlan` of `Compartments(...)` and `FlowMass(...)` |
| `model.run(parameters)` | `model.compile()` once, then `compiled.run(params, ...)` many times (jit-compiled) |

The row that matters for this project is the second-to-last: **a flow rate can depend on the
current state of the model**. That is all a prevalence trigger needs.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    EntryFlow,
    ExitFlow,
    FlowModel,
    Param,
    Property,
    PropertyMap,
    Reduce,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix

import tb_trigger.model as tbm

## 1. Compartments

A `Property` is a named set of mutually exclusive traits. A `PropertyMap` is the table of
compartments built from properties.

The model has two kinds of compartment:

* **population**: the six TB states. `S` susceptible, `E` early latent, `L` late latent, `I`
  active TB, `T` on treatment, `R` recovered or treated.
* **programme**: three bookkeeping compartments holding one unit of mass between them, which
  record whether the intervention programme is `waiting`, `scaling_up` or `active`. They are
  the trigger's memory. Notebook 2 explains them; here they only need to exist.

`stratify(..., where=...)` adds the TB states to the population compartments only, and the
programme statuses to the programme compartments only. summer2 had no equivalent; there every
compartment carried every stratification. `pop` is a one-trait grouping axis that the force of
infection groups by. It is the same pattern summer4 uses for any unstratified model.

In [ ]:
kind = Property("kind", ("population", "programme"))
state = Property("state", ("S", "E", "L", "I", "T", "R"))
programme = Property("programme", ("waiting", "scaling_up", "active"))
pop = Property("pop", ("all",))

pmap = (
    PropertyMap.from_property(kind)
    .stratify(state, where=kind["population"])
    .stratify(programme, where=kind["programme"])
    .stratify(pop)
)
pd.Series(pmap.labels(), name="compartment").to_frame()

### Selectors

Selectors pick compartments out of the map: `state["I"]`, `kind["population"]`, combined with
`&`, `|` and `~`. Every selector in this model names its `kind` explicitly, as in
`kind["population"] & state["I"]`.

That is a workaround. In `v0.2.0a5`, `Reduce(where=state["I"])` also counts every compartment
that has no `state` at all, which here means the programme compartments. Spelling out the
`kind` sidesteps the bug, and the test suite guards it. The helpers `tbm.people("I")` and
`tbm.programme("active")` build these selectors.

In [ ]:
people = kind["population"]


def P(trait):
    """A TB state among the population compartments."""
    return people & state[trait]


def prog(trait):
    """A programme status among the programme compartments."""
    return kind["programme"] & programme[trait]


{name: pmap.select(sel).tolist() for name, sel in [("people", people), ("I", P("I")), ("programme active", prog("active"))]}

## 2. Parameters

A rate written as `Param("detection")` is a placeholder that is filled from a dict when the
model runs. There are no `Parameter` objects to construct. Because parameter values are inputs
to a compiled function, one compiled model runs every scenario in this repository, and JAX
compiles it only once.

The defaults are in `tbm.DEFAULT_PARAMS`, in years:

In [ ]:
pd.Series(tbm.DEFAULT_PARAMS, name="default").to_frame()

## 3. Flows

### Transmission

Frequency-dependent transmission with partial protection against reinfection for the latently
infected (`L`) and the recovered (`R`). The force of infection is one object reused on three
flows. Multiplying it by `Param("rr_reinfection")` builds a new rate expression; it does not
copy the calculation.

The **contact rate rises** by a factor `transmission_rise` between 2020 and 2035. This is the
scenario's driver: something makes transmission worse (crowding, undernutrition, displacement)
and prevalence climbs towards the trigger threshold. `linear(Time(), knots, values)` is
summer2's linear interpolation function, and its knots and values can themselves be
parameters.

`denominator=people` keeps the programme compartments out of the population denominator.

In [ ]:
model = FlowModel(pmap)

foi = ForceOfInfection(
    "infection",
    infectious=P("I"),
    group_by=pop,
    mixing=MixingMatrix(pop, [[1.0]], check_reciprocal=False),
    contact_rate=Param("contact_rate") * tbm.transmission_multiplier(),
    denominator=people,
)
reinfection = foi * Param("rr_reinfection")
model.add_flow(TransitionFlow("infection", P("S"), P("E"), foi))
model.add_flow(TransitionFlow("reinfection_latent", P("L"), P("E"), reinfection))
model.add_flow(TransitionFlow("reinfection_recovered", P("R"), P("E"), reinfection));

### Natural history, case finding and treatment

Two-stage latency (early `E`, late `L`), self-recovery and TB death from `I`. Passive
detection moves people from `I` to `T`. Treatment ends in cure (`R`) or failure (back to `I`).

Two intervention flows are included from the start, with rates multiplied by `coverage`, the
amount of programme mass in `active`:

* `acf_detection`, **active case finding**: screening a fraction `acf_screening` of the
  population each year finds active cases with sensitivity `acf_sensitivity`.
* `tpt_early` / `tpt_late`, **preventive therapy**: the latently infected complete TPT at rate
  `tpt_rate` and are protected with probability `tpt_efficacy`.

Both default to zero strength, so in the baseline they do nothing.

In [ ]:
coverage = Reduce(sum_over=pop, where=prog("active"))

model.add_flow(TransitionFlow("stabilisation", P("E"), P("L"), Param("stabilisation")))
model.add_flow(TransitionFlow("early_activation", P("E"), P("I"), Param("early_activation")))
model.add_flow(TransitionFlow("late_activation", P("L"), P("I"), Param("late_activation")))
model.add_flow(TransitionFlow("self_recovery", P("I"), P("R"), Param("self_recovery")))

model.add_flow(TransitionFlow("detection", P("I"), P("T"), Param("detection")))
acf_rate = Param("acf_screening") * Param("acf_sensitivity") * coverage
model.add_flow(TransitionFlow("acf_detection", P("I"), P("T"), acf_rate))
per_course = 1.0 / Param("treatment_duration")
model.add_flow(
    TransitionFlow("treatment_success", P("T"), P("R"), Param("treatment_success") * per_course)
)
model.add_flow(
    TransitionFlow(
        "treatment_failure", P("T"), P("I"), (1.0 - Param("treatment_success")) * per_course
    )
)

tpt_rate = Param("tpt_rate") * Param("tpt_efficacy") * coverage
model.add_flow(TransitionFlow("tpt_early", P("E"), P("R"), tpt_rate))
model.add_flow(TransitionFlow("tpt_late", P("L"), P("R"), tpt_rate));

### Deaths and births

`add_flow` returns a handle to the flow. `tb_death.sum() + background.sum()` is a rate
expression for the total death rate. Using it as the birth rate keeps the population constant,
which is what summer2's replacement birth flow did.

In [ ]:
tb_death = model.add_flow(ExitFlow("tb_death", P("I"), Param("tb_death")))
background = model.add_flow(ExitFlow("background_death", people, Param("background_death")))
model.add_flow(EntryFlow("birth", P("S"), tb_death.sum() + background.sum()));

### The trigger (explained in notebook 2)

These three flows move the programme's unit of mass. `trigger` is fast once TB prevalence is
above `Param("threshold")` and essentially zero below it. `roll_out` takes a mean of
`rollout_years`. `stand_down` is off unless a scenario sets it.

In [ ]:
switch_on = tbm.above_threshold(
    tbm.prevalence_per_100k(), Param("threshold"), Param("threshold_width")
)
model.add_flow(
    TransitionFlow(
        "trigger", prog("waiting"), prog("scaling_up"), Param("trigger_speed") * switch_on
    )
)
model.add_flow(
    TransitionFlow("roll_out", prog("scaling_up"), prog("active"), 1.0 / Param("rollout_years"))
)
model.add_flow(
    TransitionFlow(
        "stand_down", prog("active"), prog("waiting"), Param("stand_down") * (1.0 - switch_on)
    )
)

model.set_initial_population({P("S"): 1e5 - 10.0, P("I"): 10.0, prog("waiting"): 1.0})
tbm.flow_table(model)

## 4. Compile

`compile()` turns the declarations into a JAX vector field. A compiled model compares equal to
any other model built from the same declarations, and the comparison is by value, not by
object identity. That lets us confirm that what we just built cell by cell is exactly what
`tbm.build_model()` returns. The rest of the repository uses `build_model()`.

In [ ]:
cm = model.compile()
cm == tbm.build_model().compile()

## 5. Run the baseline

The run starts in 1850 with 10 infectious people in a population of 100,000, so compartment
sizes are already "per 100,000". It burns in to endemic equilibrium and then runs to 2050. The
solver is diffrax's adaptive `tsit5`.

`tbm.save_plan(ts)` asks for every compartment and the flows behind the usual indicators.
`tbm.indicators(result)` turns the result into a per-100,000 table: prevalence, incidence,
notifications, TB deaths, and the annual risk of infection (ARI), computed from the saved
force of infection.

In [ ]:
params = tbm.scenario_params("baseline")
ts = np.round(np.arange(1850.0, 2050.05, 0.1), 1)
result = cm.run(params, save=tbm.save_plan(ts), **tbm.RUN_KWARGS)
baseline = tbm.indicators(result)
result.solver

### Burn-in and the rise

The figure should show prevalence climbing from the 1850 seed to a flat endemic level, about
235 per 100,000, well before 2020. From 2020 it rises as transmission worsens and crosses the
dashed threshold (350 per 100,000) around 2030. The overshoot and partial fall after 2040 come
from depletion of susceptibles and the slow refill of the latent pool.

In [ ]:
threshold = params["threshold"]
fig = baseline["prevalence"].plot(
    title="Baseline: active TB prevalence, burn-in from 1850",
    labels={"year": "year", "value": "per 100,000"},
)
fig.add_hline(y=threshold, line_dash="dash", annotation_text="trigger threshold")
fig.add_vrect(x0=2020, x1=2035, fillcolor="grey", opacity=0.15, annotation_text="transmission rising")
fig

### Where people are

Compartment sizes on a log scale, from 1990. The late latent pool and susceptibles dominate.
Active TB (`I`) and treatment (`T`) hold a few hundred people or fewer. After 2020 the latent
compartments grow as transmission rises.

In [ ]:
comp = result["compartments"]
names = tbm.short_names(cm.pmap)
sizes = pd.DataFrame(
    np.asarray(comp.values.data), index=np.asarray(comp.times.values).ravel(), columns=names
)
sizes.index.name = "year"
sizes.loc[1990:, ["S", "E", "L", "I", "T", "R"]].plot(
    title="Baseline compartment sizes",
    labels={"year": "year", "value": "people (of 100,000)", "variable": "compartment"},
    log_y=True,
)

### The usual indicators

Prevalence, incidence, notifications and deaths, per 100,000. All four rise together after
2020. Notifications are about 60% of incidence (passive detection rate 0.6 per year, against
0.4 per year of self-recovery and death, plus background mortality).

In [ ]:
baseline.loc[1990:, ["prevalence", "incidence", "notifications", "tb_deaths"]].plot(
    title="Baseline TB indicators",
    labels={"year": "year", "value": "per 100,000 (per year for flows)", "variable": "indicator"},
)

### What drives the rise

The annual risk of infection (ARI) comes from the force of infection that
`ForceOfInfection` captures automatically (`GroupedOutput("infection")` in the save plan).
It more than triples between 2020 and 2040. The case detection ratio (notifications divided by
incidence) stays around 60–65%. It dips while incidence rises fastest, because people
are detected some months after they become ill. Case finding itself does not change in the
baseline.

In [ ]:
drivers = pd.DataFrame(
    {
        "annual risk of infection (%)": 100 * baseline["ari"],
        "case detection ratio (%)": 100 * baseline["notifications"] / baseline["incidence"],
    }
).loc[1990:]
drivers.plot(title="Transmission and detection in the baseline", labels={"year": "year", "value": "%", "variable": ""})

### The programme switch in the baseline

The trigger is in every scenario, the baseline included. The baseline just gives the programme
nothing to do (`acf_screening = tpt_rate = 0`). The figure therefore shows **when** the trigger
would fire: the programme mass leaves `waiting` the moment prevalence crosses 350 per 100,000,
spends a few months in `scaling_up`, and ends `active`. Prevalence carries on as if nothing
happened, because nothing did.

In [ ]:
cols = ["programme_waiting", "programme_scaling_up", "programme_active"]
fig = baseline.loc[2025:2035, cols].plot(
    title="Programme status in the baseline (no intervention attached)",
    labels={"year": "year", "value": "fraction of programme", "variable": "status"},
)
fig.add_vline(x=tbm.crossing_year(baseline["prevalence"], threshold), line_dash="dash", annotation_text="prevalence crosses threshold")
fig

## Try it

* Change `transmission_rise` (for example `1.4` or `2.2`) in `tbm.scenario_params("baseline",
  transmission_rise=...)` and rerun section 5. The crossing year moves, or prevalence never
  crosses at all.
* Change `detection` to see the endemic level and the case detection ratio move together.

Next: [`02-triggered-interventions`](02-triggered-interventions.ipynb).